# Laboratorio 4: Árboles de decisión
Consultoría para SmartStay Advisors

In [9]:
# Librerias a utilizar

import importlib, subprocess, sys

required = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "pyreadr": "pyreadr"
}

for module, pkg in required.items():
    try:
        importlib.import_module(module)
    except ImportError:
        print(f"Instalando {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])
        importlib.import_module(module)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyreadr
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

## Actividades: 

1. Use los mismos conjuntos de entrenamiento y prueba que usó para los modelos de regresión lineal en la entrega anterior.

In [ ]:
# Cargar el dataset
resultado = pyreadr.read_r('listings.RData')
nombre_objeto = list(resultado.keys())[0]
df = resultado[nombre_objeto]

# Limpieza y normalización de price
df['price'] = df['price'].astype(str).str.replace(r'[^\d\.\-]', '', regex=True)

# Convertir a numérico
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Eliminar filas con price NaN
n_before = len(df)
mask_valid_price = df['price'].notna() & (df['price'] > 0)
df_clean = df.loc[mask_valid_price].copy()
n_after = len(df_clean)
print(f"Filas totales: {n_before}, después limpieza price: {n_after} (eliminadas {n_before-n_after})")

# Transformación y selección de variables
df_model = df_clean.copy()
df_model['log_price'] = np.log(df_model['price'])

# Asegurarnos de que las variables sean numéricas
for col in ['accommodates', 'bedrooms', 'beds', 'number_of_reviews', 'review_scores_rating']:
    if col not in df_model.columns:
        df_model[col] = np.nan
    df_model[col] = pd.to_numeric(df_model[col], errors='coerce')

# Selección de características
features = ['accommodates', 'bedrooms', 'beds', 'number_of_reviews', 'review_scores_rating']
X = df_model[features].copy()

# Agregamos room_type con One-Hot Encoding si existe
if 'room_type' in df_model.columns:
    dummies = pd.get_dummies(df_model['room_type'], drop_first=True)
    X = pd.concat([X, dummies], axis=1)

y = df_model['log_price'].copy()

# Imputación de nulos con la mediana
X = X.fillna(X.median())

Filas totales: 171748, después limpieza price: 76246 (eliminadas 95502)
